# BODAQS Session Browser - Study Set Scope

This notebook opens the session/GPS browser against a saved Study Set created by the web app, library manager, or another Library API client.

It deliberately does not include an in-notebook session selector. The Study Set defines the available session pool; the browser widget then lets you inspect one session at a time.

## 1. Configure Library And Study Set

Set `LIBRARIES_ROOT`, `LIBRARY_ID`, and `TARGET_STUDY_SET_ID` before running the dashboard cell. Leave `TARGET_STUDY_SET_ID` blank to list available Study Sets for the configured library.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd
import plotly.io as pio


def find_analysis_dir(start: Path | None = None) -> Path:
    """Find the analysis package root whether Jupyter starts in repo root or analysis/."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        analysis = candidate / "analysis"
        if (analysis / "bodaqs_analysis").is_dir():
            return analysis
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path(r"C:\Users\benco\OneDrive\BODAQS-data")
LIBRARY_ID = "archie"
TARGET_STUDY_SET_ID = "archie-evedon-26_v2"  # Example: "ben-stevo-test-rides"

# The session browser inspects physical sessions, so Study Set groupings are
# not needed in the selector bridge for this notebook.
INCLUDE_STUDY_SET_GROUPINGS = False

pio.renderers.default = "notebook_connected"

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Libraries root: {LIBRARIES_ROOT}")


Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Libraries root: C:\Users\benco\OneDrive\BODAQS-data


## 2. Load Study Set Scope

In [2]:
from bodaqs_analysis.library_api import InvalidStudySetError, LibraryAdapter, make_study_set_selector_handle

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

available_study_sets = adapter.list_study_sets(library_id=LIBRARY_ID)
if not TARGET_STUDY_SET_ID.strip():
    if available_study_sets:
        display(pd.DataFrame(available_study_sets))
    else:
        print("No Study Sets found for this library.")
    raise ValueError("Set TARGET_STUDY_SET_ID to a saved Study Set ID, then rerun this cell.")

try:
    study_set_bridge = adapter.study_set_to_selection_snapshot(
        LIBRARY_ID,
        TARGET_STUDY_SET_ID.strip(),
        include_groupings=INCLUDE_STUDY_SET_GROUPINGS,
    )
except InvalidStudySetError as exc:
    message = str(exc)
    if "one-library Study Sets" in message:
        raise RuntimeError(
            "This pilot notebook currently supports one-library Study Sets only. "
            "Open a Study Set whose sessions all belong to LIBRARY_ID, or wait for the multi-library notebook bridge."
        ) from exc
    raise

sel = make_study_set_selector_handle(
    study_set_bridge,
    title="Sessions to chart",
    rows=8,
    select_first_by_default=True,
)
key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()

print(f"Loaded Study Set: {study_set_bridge['display_name']} ({study_set_bridge['study_set_id']})")
print(f"Sessions selected for charting: {len(key_to_ref)}")
display(sel["ui"])
display(events_index_df)


Loaded Study Set: Archie-Evedon-26_v2 (archie-evedon-26-v2)
Sessions selected for charting: 1


,session_key,run_id,session_id
0,archie-mega-local_260617_112959::260613_133202,archie-mega-local_260617_112959,260613_133202


## 3. Open Session Browser

In [3]:
from bodaqs_analysis.dashboards import make_session_gps_dashboard

if not sel["get_key_to_ref"]():
    raise ValueError("Select at least one Study Set session before opening the dashboard.")

dashboard = make_session_gps_dashboard(sel)
display(dashboard["ui"])
